## CSC311 Project: Model Training

This notebook trains a Multinomial Logistic Regression model on the project dataset. It performs the following steps:
1.  Loads the data and identifies feature types.
2.  Splits the data into training and validation sets, ensuring responses from the same user stay in the same set.
3.  Builds a preprocessing pipeline using `scikit-learn` that:
    *   Imputes missing values.
    *   Applies TF-IDF to text features.
    *   One-hot encodes categorical features.
    *   Standardizes the final feature matrix.
4.  Trains a `LogisticRegression` model.
5.  Evaluates the model on the validation set.
6.  Exports all necessary assets (model weights, preprocessing stats, etc.) to JSON and NumPy files for use in `pred.py`.

In [1]:
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

### 1. Load Data and Define Features

In [2]:
# Load data
df = pd.read_csv('training_data_202601.csv')

# Rename columns for easier access
df.rename(columns={
    'Painting': 'artist',
    'unique_id': 'user_id',
    'On a scale of 1–10, how intense is the emotion conveyed by the artwork?': 'impression_rating',
    'How much (in Canadian dollars) would you be willing to pay for this painting?': 'price',
    'What season does this art piece remind you of?': 'season',
    'If you could purchase this painting, which room would you put that painting in?': 'room',
    'Describe how this painting makes you feel.': 'moods_text',
    'Imagine a soundtrack for this painting. Describe that soundtrack without naming any objects in the painting.': 'story_text'
}, inplace=True)

# Force numerical columns to numeric type, coercing errors to NaN
for col in ['impression_rating', 'price']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Define feature types based on the new column names
TARGET = 'artist'

# Note: I'm selecting a subset of numerical features that seem most relevant.
# The original 'like_rating', 'interest_rating', 'overall_rating' are not in this CSV.
NUMERICAL_FEATURES = [
    'impression_rating',
    'price'
]

# Note: 'time_of_day' is not in this CSV.
CATEGORICAL_FEATURES = [
    'season',
    'room'
]

TEXT_FEATURES = [
    'moods_text',
    'story_text'
]

# For simplicity, we'll combine the text features into one
df['combined_text'] = df[TEXT_FEATURES].fillna('').agg(' '.join, axis=1)
SINGLE_TEXT_FEATURE = 'combined_text'

### 2. Grouped Train-Validation Split

We split by `user_id` to prevent data leakage.

In [3]:
user_ids = df['user_id'].unique()
train_user_ids, val_user_ids = train_test_split(user_ids, test_size=0.2, random_state=42)

train_df = df[df['user_id'].isin(train_user_ids)]
val_df = df[df['user_id'].isin(val_user_ids)]

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_val = val_df.drop(columns=[TARGET])
y_val = val_df[TARGET]

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")

Training set size: 1347
Validation set size: 339


### 3. Build Preprocessing and Model Pipeline

In [4]:
# Create preprocessing pipelines for each feature type
numeric_transformer = SimpleImputer(strategy='mean')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
text_transformer = TfidfVectorizer(max_features=500, stop_words='english')

# Create a preprocessor object using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, NUMERICAL_FEATURES),
        ('cat', categorical_transformer, CATEGORICAL_FEATURES),
        ('text', text_transformer, SINGLE_TEXT_FEATURE)
    ],
    remainder='drop'
)

# Create the full model pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler(with_mean=False)), # Use with_mean=False for sparse matrices from TF-IDF
    ('classifier', LogisticRegression(solver='lbfgs', C=1.0, random_state=42, max_iter=1000))
])

### 4. Train and Evaluate the Model

In [5]:
model_pipeline.fit(X_train, y_train)

# Evaluate on validation set
y_pred = model_pipeline.predict(X_val)
print(classification_report(y_val, y_pred))

                           precision    recall  f1-score   support

The Persistence of Memory       0.84      0.81      0.83       113
         The Starry Night       0.73      0.72      0.72       113
      The Water Lily Pond       0.79      0.82      0.81       113

                 accuracy                           0.78       339
                macro avg       0.78      0.78      0.78       339
             weighted avg       0.78      0.78      0.78       339



### 5. Export Model Assets for `pred.py`

This is the crucial step. We extract all the learned parameters and save them.

In [6]:
# --- Extract Model Parameters ---
classifier = model_pipeline.named_steps['classifier']
model_params = {
    'weights': classifier.coef_.tolist(),
    'intercept': classifier.intercept_.tolist(),
    'classes': classifier.classes_.tolist()
}
with open('model_params.json', 'w') as f:
    json.dump(model_params, f)

# --- Extract Preprocessing Statistics ---
preprocessor_step = model_pipeline.named_steps['preprocessor']
scaler_step = model_pipeline.named_steps['scaler']

# Imputation values
imputation_means = dict(zip(NUMERICAL_FEATURES, preprocessor_step.named_transformers_['num'].statistics_))
imputation_modes = dict(zip(CATEGORICAL_FEATURES, preprocessor_step.named_transformers_['cat'].named_steps['imputer'].statistics_))

# One-hot encoder categories
ohe_categories = {col: cats.tolist() for col, cats in zip(CATEGORICAL_FEATURES, preprocessor_step.named_transformers_['cat'].named_steps['onehot'].categories_)}

# TF-IDF vocabulary and IDF scores
tfidf_vocab = preprocessor_step.named_transformers_['text'].get_feature_names_out().tolist()
tfidf_idf = preprocessor_step.named_transformers_['text'].idf_.tolist()
with open('tfidf_vocab.json', 'w') as f:
    json.dump(tfidf_vocab, f)

# Feature scaling values (mean and std)
# Note: We need to reconstruct the full feature order
ohe_feature_names = preprocessor_step.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(CATEGORICAL_FEATURES).tolist()
all_feature_names = NUMERICAL_FEATURES + ohe_feature_names + tfidf_vocab

preprocessing_stats = {
    'imputation_means': imputation_means,
    'imputation_modes': imputation_modes,
    'numerical_features': NUMERICAL_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'text_features': [SINGLE_TEXT_FEATURE], # Use the combined one
    'one_hot_categories': ohe_categories,
    'feature_means': scaler_step.mean_.tolist(),
    'feature_stds': scaler_step.scale_.tolist(),
    'tfidf_idf': tfidf_idf
}

with open('preprocessing_stats.json', 'w') as f:
    json.dump(preprocessing_stats, f, indent=4)

print("Model assets exported successfully.")

Model assets exported successfully.
